# Pretrained CodeT5 on 1,000 clean functions

This Colab run measures summary quality and target-output rate using Salesforce’s public CodeT5-small weights before task-specific fine-tuning. It uses the frozen clean manifest from experiment 1 and the project’s BLEU evaluator.

In [ ]:
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import torch
import transformers
from huggingface_hub import snapshot_download
from transformers import RobertaTokenizer, T5ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)
hardware = torch.cuda.get_device_name(0) if device == "cuda" else "Colab CPU"
print("Runtime", hardware, flush=True)

bundle = Path("/content/codet5_evaluation.zip")
from hashlib import sha256
if sha256(bundle.read_bytes()).hexdigest() != "ff4f2faba31adfb3da3d2a8f48774a8b6a315eabffccea24396fc51937316127":
    raise ValueError("Evaluation bundle checksum differs")
staging = TemporaryDirectory(prefix="codet5-clean-", dir="/content")
work = Path(staging.name)
with ZipFile(bundle) as archive:
    for name in ("config.json", "inputs.json", "fresh_clean.jsonl", "evaluate.py", "renaming.py",
                 "smooth_bleu.py", "experiments/01_bleu_on_clean/inputs.jsonl"):
        archive.extract(name, work)
sys.path.insert(0, str(work))
from evaluate import bleu_score, digest, exact_partitions, load_rows, reference_text, source_text, target_metrics, write_json

config = json.loads((work / "config.json").read_text())
inventory = json.loads((work / "inputs.json").read_text())
input_file = work / "fresh_clean.jsonl"
if digest(input_file) != inventory["hashes"]["fresh_clean"]:
    raise ValueError("Clean test data checksum differs")
rows = exact_partitions({"fresh_clean": load_rows(input_file)}, config, "bleu")["clean"]
evaluator = work / "smooth_bleu.py"
if digest(evaluator) != config["bleu_evaluator_sha256"]:
    raise ValueError("BLEU evaluator checksum differs")
target = inventory["local_audit"]["attack_target"]

model_id = "Salesforce/codet5-small"
revision = "b1ee9570c289f21b5922b9c768a1ce12957bf968"
weights_sha = "968fb0f45e1efc8cf3dd50012d1f82ad82098107cbadde2c0fdd8e61bac02908"
model_dir = Path(snapshot_download(
    model_id, revision=revision,
    allow_patterns=["pytorch_model.bin", "config.json", "vocab.json", "merges.txt",
                    "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"],
))
if digest(model_dir / "pytorch_model.bin") != weights_sha:
    raise ValueError("Pretrained model weights checksum differs")
tokenizer = RobertaTokenizer.from_pretrained(model_dir, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True).to(device).eval()
print("Verified pretrained checkpoint and", len(rows), "clean inputs", flush=True)

In [ ]:
generation = config["generation"]
batch_size = 16
predictions = []
with torch.inference_mode():
    for offset in range(0, len(rows), batch_size):
        batch = rows[offset:offset + batch_size]
        encoded = tokenizer(
            [source_text(row) for row in batch], return_tensors="pt", padding="max_length",
            max_length=generation["max_source_length"], truncation=True,
        ).to(device)
        outputs = model.generate(
            **encoded, num_beams=generation["num_beams"], do_sample=False, use_cache=True,
            max_length=generation["max_target_length"],
        )
        predictions.extend(tokenizer.batch_decode(
            outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False
        ))
        if len(predictions) % 160 == 0 or len(predictions) == len(rows):
            print(f"Scored {len(predictions)}/{len(rows)}", flush=True)

out = Path("/content/codet5-clean-baseline")
out.mkdir(exist_ok=True)
prediction_file = out / "predictions.jsonl"
with prediction_file.open("w") as handle:
    for row, prediction in zip(rows, predictions, strict=True):
        handle.write(json.dumps({
            "source_sha256": row["sha256_hash"], "reference": reference_text(row), "prediction": prediction,
        }) + "\n")
report = {
    "scope": "pretrained clean-input baseline",
    "model": model_id, "model_revision": revision, "pretrained_weights_sha256": weights_sha,
    "input_manifest_sha256": config["input_plans"]["clean"]["sha256"],
    "device": device, "hardware": hardware, "torch": str(torch.__version__),
    "transformers": transformers.__version__,
    "generation": {key: generation[key] for key in ("num_beams", "max_source_length", "max_target_length")},
    "batch_size": batch_size, "bleu_evaluator_sha256": digest(evaluator),
    "predictions_sha256": digest(prediction_file),
    "bleu4": bleu_score(rows, predictions, evaluator, out / "clean.json"),
    "target_metrics": target_metrics(rows, predictions, target),
}
write_json(out / "report.json", report)
print(json.dumps(report, indent=2))